In [1]:
# ============================================================
# NB10_Quarterly_Backtest_v2   (rename to NB09 only if that name is free)
#
# Walk-forward backtest of SLOW_GROWER and STALWART screens.
#
# Variants run on the SAME screen dates, so the comparison is fair:
#   ANNUAL_ONLY_90D      annual filters only, 90-day hold
#   QUARTERLY_GATED_90D  annual filters + quarterly EPS/sales gates, 90-day hold
#   ANNUAL_ONLY_365D     annual filters only, 365-day hold, yearly screen dates
#
# Point-in-time rules:
#   quarterly statements usable QTR_LAG days after period end (default 45)
#   annual statements usable ANN_LAG days after period end (default 90)
#   dividends use ex-dates up to the screen date only
#
# Honest limits (print them in any write-up):
#   * Yahoo gives ~4 annual and ~6 quarterly periods, so the quarterly YoY
#     gate only has data for the most recent screens. Check
#     backtest_v2_screens.with_qtr_yoy before reading anything into results.
#   * Universe is today's list (survivorship bias); shares outstanding are
#     current values; stop-loss is checked on daily closes only.
#   * Exploratory and directional only. Not investment advice.
# ============================================================

import re
import uuid
import numpy as np
import pandas as pd
from datetime import datetime
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType, DoubleType, LongType
)

RUN_ID = str(uuid.uuid4())
RUN_TS = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

# ------------------------------------------------------------
# 1. Config (same file and keys as NB03; new keys have defaults)
# ------------------------------------------------------------
try:
    params = (
        spark.read.format("csv")
        .option("header", "true")
        .load("Files/config/lynch_parameters.csv")
        .toPandas()
        .set_index("parameter")["value"]
        .to_dict()
    )
except Exception as e:
    print(f"Could not read config; using defaults. Details: {e}")
    params = {}

def pf(name, default):
    try:
        return float(params.get(name, default))
    except (TypeError, ValueError):
        return float(default)

def ps(name, default):
    v = params.get(name, default)
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return default
    return str(v).strip()

# Style bands: identical to NB03
SLOW_MIN, SLOW_MAX = pf("slow_eps_cagr_min", 0.03), pf("slow_eps_cagr_max", 0.12)
STAL_MIN, STAL_MAX = pf("stalwart_eps_cagr_min", 0.08), pf("stalwart_eps_cagr_max", 0.16)
# NB03 checks Slow first, so 8-12% EPS CAGR is classed Slow Grower.
# The effective Stalwart band is therefore max(STAL_MIN, SLOW_MAX) to STAL_MAX.
STAL_LOW_EFFECTIVE = max(STAL_MIN, SLOW_MAX)

MIN_MCAP_CR   = pf("min_market_cap_cr", 2000)
SG_YIELD_MIN  = pf("slow_grower_yield_min", 0.03)
SG_PAYOUT_MAX = pf("slow_grower_payout_max", 0.80)
SG_DEBT_MAX   = pf("slow_grower_debt_max", 0.75)
SG_Q_EPS_FLOOR = pf("slow_grower_quarterly_eps_yoy_floor", -0.20)
ST_DEBT_MAX   = pf("stalwart_debt_max", 0.50)
ST_Q_EPS_MIN  = pf("quarterly_eps_yoy_min", 0.10)
ST_Q_SAL_MIN  = pf("quarterly_sales_yoy_min", 0.08)

TOP_N         = int(pf("top_n_candidates", 15))
MIN_UNIVERSE  = int(pf("min_universe_size", 50))
TXN_COST      = pf("transaction_cost_pct", 0.5) / 100.0   # 0.5 means 0.5%
STOP          = pf("stop_loss_pct", 0.07)                 # 0 disables
TAKE          = pf("take_profit_pct", 0.0)                # 0 disables
HOLD_Q        = int(pf("holding_period_days_quarterly", 90))
HOLD_Y        = int(pf("holding_period_days_yearly", 365))
QTR_LAG       = int(pf("quarterly_publication_lag_days", 45))
ANN_LAG       = int(pf("annual_publication_lag_days", 90))
BENCH         = ps("benchmark_symbol", "NIFTY50")
ANNUAL_MONTHS = [int(x) for x in ps("annual_screen_months", "7").split(";") if x.strip()]

# ------------------------------------------------------------
# 2. Load data
# ------------------------------------------------------------
universe = (
    spark.table("bronze_universe")
    .filter("in_scope = true")
    .select("symbol", "company_name", "industry")
    .dropDuplicates(["symbol"])
    .toPandas()
)
if universe.empty:
    raise RuntimeError("bronze_universe has no in-scope rows.")

fund = spark.table("bronze_yahoo_fundamentals_long").toPandas()
prices = spark.table("bronze_yahoo_prices").select("symbol", "trade_date", "close").toPandas()
dividends = spark.table("bronze_yahoo_dividends").toPandas()
market = spark.table("bronze_yahoo_market_snapshot").toPandas()

fund["period_type"] = fund["period_type"].astype(str).str.upper()
fund["statement_type"] = fund["statement_type"].astype(str).str.upper()
fund["period_end"] = pd.to_datetime(fund["period_end"], errors="coerce")
fund["value"] = pd.to_numeric(fund["value"], errors="coerce")
fund = fund.dropna(subset=["symbol", "period_end", "value"])

prices["trade_date"] = pd.to_datetime(prices["trade_date"], errors="coerce")
prices["close"] = pd.to_numeric(prices["close"], errors="coerce")
prices = prices.dropna(subset=["symbol", "trade_date", "close"])
prices = prices[prices["close"] > 0]
prices = prices.drop_duplicates(["symbol", "trade_date"]).sort_values(["symbol", "trade_date"])
LAST_PX_DATE = prices["trade_date"].max()

dividends["ex_date"] = pd.to_datetime(dividends["ex_date"], errors="coerce")
dividends["dividend"] = pd.to_numeric(dividends["dividend"], errors="coerce")
dividends = dividends.dropna(subset=["symbol", "ex_date", "dividend"])
dividends = dividends[dividends["dividend"] > 0].drop_duplicates(["symbol", "ex_date"])

market = market.drop_duplicates(["symbol"], keep="last")
MKT = {
    str(r["symbol"]): {
        "shares": pd.to_numeric(r.get("shares_outstanding"), errors="coerce"),
    }
    for _, r in market.iterrows()
}

# Same Yahoo-field mapping as NB03 (keep in sync, or move to a shared %run notebook)
def normalize_label(value):
    return re.sub(r"[^a-z0-9]", "", str(value).lower())

def canonical_metric(source_field, statement_type):
    key = normalize_label(source_field)
    st = str(statement_type).upper()
    if "INCOME" in st:
        if key in {"dilutedeps", "dilutedepscontinuingoperations", "dilutedepscontinuinganddiscontinuedoperations"}:
            return "eps_diluted"
        if key in {"basiceps", "basicepscontinuingoperations"}:
            return "eps_basic"
        if key in {"totalrevenue", "operatingrevenue", "totalrevenuefromoperations", "revenue"}:
            return "revenue"
        if key in {"ebit", "operatingincome"}:
            return "ebit"
        if key in {"netincome", "netincomecommonstockholders", "netincomecommonstockholdersincludingextraordinaryitems", "netincomecontinuingoperations"}:
            return "net_income"
        if "interestexpense" in key:
            return "interest_expense"
    if "BALANCE" in st:
        if key in {"stockholdersequity", "commonstockequity", "totalequitygrossminorityinterest", "totalequity"}:
            return "equity"
        if key == "totaldebt":
            return "total_debt"
        if key in {"longtermdebt", "longtermdebtandcapitalleaseobligation"}:
            return "long_debt"
        if key in {"currentdebt", "shorttermdebt", "currentdebtandcapitalleaseobligation"}:
            return "short_debt"
        if key in {"ordinarysharesnumber", "shareissued"}:
            return "shares"
    if "CASH" in st or "FLOW" in st:
        if key in {"operatingcashflow", "cashflowfromcontinuingoperatingactivities", "cashfromoperatingactivities"}:
            return "ocf"
    return None

def source_priority(source_field, metric):
    key = normalize_label(source_field)
    if metric == "eps":
        return 0 if "diluted" in key else 1
    if metric == "total_debt":
        return 0 if key == "totaldebt" else 1
    if metric == "equity":
        if key == "stockholdersequity": return 0
        if key == "commonstockequity": return 1
        return 2
    return 0

fund["metric"] = fund.apply(lambda r: canonical_metric(r["source_field"], r["statement_type"]), axis=1)
fund = fund.dropna(subset=["metric"]).copy()
fund.loc[fund["metric"].isin(["eps_diluted", "eps_basic"]), "metric"] = "eps"
fund["priority"] = fund.apply(lambda r: source_priority(r["source_field"], r["metric"]), axis=1)
fund = fund.sort_values(["symbol", "period_type", "period_end", "metric", "priority"]).drop_duplicates(
    ["symbol", "period_type", "period_end", "metric"], keep="first"
)

print(f"Universe: {len(universe)} | fundamentals: {len(fund):,} | prices: {len(prices):,} | last price: {LAST_PX_DATE.date()}")

# ------------------------------------------------------------
# 3. Fast lookup structures (numpy arrays per symbol)
# ------------------------------------------------------------
SER = {}
for (sym, pt, met), g in fund.groupby(["symbol", "period_type", "metric"]):
    g = g.sort_values("period_end")
    SER[(str(sym), pt, met)] = (g["period_end"].values, g["value"].values.astype(float))

PX = {str(s): (g["trade_date"].values, g["close"].values.astype(float)) for s, g in prices.groupby("symbol")}
DIV = {str(s): (g["ex_date"].values, g["dividend"].values.astype(float)) for s, g in dividends.groupby("symbol")}

def days_between(a, b):
    """Whole days from numpy datetime64 a to b (b - a)."""
    return float((b - a).astype("timedelta64[D]").astype(float))

def px_on_or_after(sym, target, tol_days):
    d = PX.get(sym)
    if d is None:
        return None, None
    dates, closes = d
    i = int(np.searchsorted(dates, target.to_datetime64()))
    if i >= len(dates):
        return None, None
    if days_between(target.to_datetime64(), dates[i]) > tol_days:
        return None, None
    return pd.Timestamp(dates[i]), float(closes[i])

def series_upto(sym, ptype, metric, cutoff):
    s = SER.get((sym, ptype, metric))
    if s is None:
        return None
    n = int(np.searchsorted(s[0], cutoff.to_datetime64(), side="right"))
    return (s[0][:n], s[1][:n]) if n > 0 else None

def latest_val(sym, metric, q_cut, a_cut):
    """Latest value across quarterly and annual statements available at the cutoffs."""
    best = None
    for pt, cut in (("QUARTERLY", q_cut), ("ANNUAL", a_cut)):
        s = series_upto(sym, pt, metric, cut)
        if s is not None and (best is None or s[0][-1] > best[0]):
            best = (s[0][-1], float(s[1][-1]))
    return None if best is None else best[1]

# ------------------------------------------------------------
# 4. Point-in-time feature builders
# ------------------------------------------------------------
def annual_eps_cagr(sym, a_cut):
    s = series_upto(sym, "ANNUAL", "eps", a_cut)
    if s is None or len(s[1]) < 4:
        return None
    d, v = s[0][-4:], s[1][-4:]
    yrs = days_between(d[0], d[-1]) / 365.25
    if v[0] <= 0 or v[-1] <= 0 or yrs <= 0:
        return None
    return float((v[-1] / v[0]) ** (1 / yrs) - 1)

def ttm_eps(sym, q_cut, a_cut):
    s = series_upto(sym, "QUARTERLY", "eps", q_cut)
    if s is not None and len(s[1]) >= 4:
        d, v = s[0][-4:], s[1][-4:]
        span = days_between(d[0], d[-1])
        if 250 <= span <= 430 and float(v.sum()) > 0:
            return float(v.sum())
    a = series_upto(sym, "ANNUAL", "eps", a_cut)
    if a is not None and a[1][-1] > 0:
        return float(a[1][-1])
    return None

def quarterly_yoy(sym, metric, q_cut):
    """Latest quarter vs same quarter last year (matched by ~365 day gap)."""
    s = series_upto(sym, "QUARTERLY", metric, q_cut)
    if s is None:
        return None
    d, v = s
    if days_between(d[-1], q_cut.to_datetime64()) > 135:     # stale latest quarter
        return None
    gaps = np.array([days_between(x, d[-1]) for x in d])
    diff = np.abs(gaps - 365)
    j = int(np.argmin(diff))
    if diff[j] > 20 or v[j] <= 0:
        return None
    return float(v[-1] / v[j] - 1)

def debt_equity(sym, q_cut, a_cut):
    eq = latest_val(sym, "equity", q_cut, a_cut)
    debt = latest_val(sym, "total_debt", q_cut, a_cut)
    if debt is None:
        debt = (latest_val(sym, "long_debt", q_cut, a_cut) or 0.0) + (latest_val(sym, "short_debt", q_cut, a_cut) or 0.0)
    return (debt / eq) if eq is not None and eq > 0 else None

def interest_cover(sym, a_cut):
    e = series_upto(sym, "ANNUAL", "ebit", a_cut)
    i = series_upto(sym, "ANNUAL", "interest_expense", a_cut)
    if e is None or i is None or abs(i[1][-1]) <= 0:
        return None
    return float(e[1][-1] / abs(i[1][-1]))

def ocf_to_ni(sym, a_cut):
    o = series_upto(sym, "ANNUAL", "ocf", a_cut)
    n = series_upto(sym, "ANNUAL", "net_income", a_cut)
    if o is None or n is None:
        return None
    od = dict(zip(o[0], o[1]))
    ocf_sum = ni_sum = 0.0
    cnt = 0
    for dt, ni in zip(n[0][-4:], n[1][-4:]):
        if ni > 0 and dt in od:
            ocf_sum += od[dt]
            ni_sum += ni
            cnt += 1
    return float(ocf_sum / ni_sum) if cnt >= 3 and ni_sum > 0 else None

def dividend_windows(sym, D):
    """DPS in the last three 365-day windows: [ttm, previous year, year before]."""
    out = [0.0, 0.0, 0.0]
    d = DIV.get(sym)
    if d is None:
        return out
    dates, amt = d
    for k in range(3):
        hi = (D - pd.Timedelta(days=365 * k)).to_datetime64()
        lo = (D - pd.Timedelta(days=365 * (k + 1))).to_datetime64()
        out[k] = float(amt[(dates > lo) & (dates <= hi)].sum())
    return out

def build_features(D):
    q_cut = D - pd.Timedelta(days=QTR_LAG)
    a_cut = D - pd.Timedelta(days=ANN_LAG)
    rows = []
    for _, u in universe.iterrows():
        sym = str(u["symbol"])
        entry_date, entry_px = px_on_or_after(sym, D, 10)
        if entry_px is None:
            continue

        cagr = annual_eps_cagr(sym, a_cut)
        eps_ttm = ttm_eps(sym, q_cut, a_cut)
        pe = entry_px / eps_ttm if eps_ttm else None

        w = dividend_windows(sym, D)
        ttm_yield = w[0] / entry_px
        # "no cut" with 10% tolerance because rolling windows shift payment timing
        div_streak = bool(w[0] > 0 and w[1] > 0 and w[2] > 0 and w[0] >= 0.9 * w[1] and w[1] >= 0.9 * w[2])

        shares = latest_val(sym, "shares", q_cut, a_cut)
        if shares is None:
            sh = MKT.get(sym, {}).get("shares")
            shares = float(sh) if sh is not None and pd.notna(sh) else None
        mcap_cr = (shares * entry_px / 1e7) if shares else None

        ni = series_upto(sym, "ANNUAL", "net_income", a_cut)
        ann_ni = float(ni[1][-1]) if ni is not None else None
        payout = (w[0] * shares / ann_ni) if shares and ann_ni and ann_ni > 0 else None

        peg = None
        if cagr is not None and pe and pe > 0:
            g = cagr * 100 + ttm_yield * 100
            peg = g / pe if g > 0 else None

        rows.append({
            "symbol": sym,
            "company_name": u["company_name"],
            "industry": u["industry"],
            "entry_date": entry_date,
            "entry_price": entry_px,
            "cagr_3y": cagr,
            "pe": pe,
            "ttm_yield": ttm_yield,
            "div_streak": div_streak,
            "payout": payout,
            "de": debt_equity(sym, q_cut, a_cut),
            "int_cov": interest_cover(sym, a_cut),
            "ocf_ni": ocf_to_ni(sym, a_cut),
            "qtr_eps_yoy": quarterly_yoy(sym, "eps", q_cut),
            "qtr_sales_yoy": quarterly_yoy(sym, "revenue", q_cut),
            "mcap_cr": mcap_cr,
            "peg": peg,
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df
    for c in ["cagr_3y", "pe", "ttm_yield", "payout", "de", "int_cov", "ocf_ni",
              "qtr_eps_yoy", "qtr_sales_yoy", "mcap_cr", "peg", "entry_price"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["div_streak"] = df["div_streak"].astype(bool)
    return df

# ------------------------------------------------------------
# 5. Style pickers
# ------------------------------------------------------------
def cover_ok(df, minimum):
    # Missing interest expense is accepted only for almost debt-free companies
    return (df["int_cov"] >= minimum) | (df["int_cov"].isna() & (df["de"] <= 0.10))

def pick_slow_grower(df, use_qtr):
    """Dividend-first (matches NB04): rank by TTM yield, then lower P/E."""
    m = (
        (df["cagr_3y"] >= SLOW_MIN) & (df["cagr_3y"] < SLOW_MAX)
        & (df["de"] <= SG_DEBT_MAX)
        & (df["ocf_ni"] >= 0.80)
        & cover_ok(df, 4.0)
        & (df["ttm_yield"] >= SG_YIELD_MIN)
        & (df["payout"] <= SG_PAYOUT_MAX)
        & df["div_streak"]
        & (df["mcap_cr"].isna() | (df["mcap_cr"] >= MIN_MCAP_CR))
    )
    if use_qtr:
        m &= df["qtr_eps_yoy"] > SG_Q_EPS_FLOOR          # missing data fails (strict)
    d = df[m]
    return d.sort_values(["ttm_yield", "pe"], ascending=[False, True], na_position="last").head(TOP_N)

def pick_stalwart(df, use_qtr):
    """Growth at a fair price: rank by dividend-adjusted PEG (high first), then lower P/E."""
    m = (
        (df["cagr_3y"] >= STAL_LOW_EFFECTIVE) & (df["cagr_3y"] <= STAL_MAX)
        & (df["de"] <= ST_DEBT_MAX)
        & (df["ocf_ni"] >= 0.80)
        & cover_ok(df, 5.0)
        & df["peg"].notna()
        & (df["mcap_cr"].isna() | (df["mcap_cr"] >= MIN_MCAP_CR))
    )
    if use_qtr:
        m &= (df["qtr_eps_yoy"] >= ST_Q_EPS_MIN) & (df["qtr_sales_yoy"] >= ST_Q_SAL_MIN)
    d = df[m]
    return d.sort_values(["peg", "pe"], ascending=[False, True], na_position="last").head(TOP_N)

STYLES = [("SLOW_GROWER", pick_slow_grower), ("STALWART", pick_stalwart)]

VARIANTS = [
    {"name": "ANNUAL_ONLY_90D",     "use_qtr": False, "hold": HOLD_Q, "yearly_dates_only": False},
    {"name": "QUARTERLY_GATED_90D", "use_qtr": True,  "hold": HOLD_Q, "yearly_dates_only": False},
    {"name": "ANNUAL_ONLY_365D",    "use_qtr": False, "hold": HOLD_Y, "yearly_dates_only": True},
]

# ------------------------------------------------------------
# 6. Trade simulation (daily-close stop / take-profit)
# ------------------------------------------------------------
def simulate(sym, entry_date, entry_px, exit_target):
    dates, closes = PX[sym]
    i0 = int(np.searchsorted(dates, entry_date.to_datetime64()))
    i1 = int(np.searchsorted(dates, exit_target.to_datetime64()))
    if i1 >= len(dates) or days_between(exit_target.to_datetime64(), dates[i1]) > 30:
        return None
    hold_ret = float(closes[i1] / entry_px - 1)
    for j in range(i0 + 1, i1 + 1):
        r = closes[j] / entry_px - 1
        if STOP > 0 and r <= -STOP:
            return pd.Timestamp(dates[j]), float(closes[j]), "STOP", hold_ret
        if TAKE > 0 and r >= TAKE:
            return pd.Timestamp(dates[j]), float(closes[j]), "TAKE_PROFIT", hold_ret
    return pd.Timestamp(dates[i1]), float(closes[i1]), "TIME", hold_ret

def bench_return(D, hold):
    if BENCH not in PX:
        return None
    _, p0 = px_on_or_after(BENCH, D, 15)
    _, p1 = px_on_or_after(BENCH, D + pd.Timedelta(days=hold), 30)
    return (p1 / p0 - 1) if p0 and p1 else None

# ------------------------------------------------------------
# 7. Screen dates
# ------------------------------------------------------------
start = pd.to_datetime(ps("backtest_start_date", "2025-07-01"))
end_cfg = ps("backtest_end_date", "")
end = pd.to_datetime(end_cfg) if end_cfg else LAST_PX_DATE - pd.Timedelta(days=HOLD_Q)

explicit = ps("backtest_screen_dates", "")        # e.g. 2025-08-01;2025-11-01;2026-02-01;2026-05-01
if explicit:
    screen_dates = [pd.Timestamp(x.strip()) for x in explicit.split(";") if x.strip()]
else:
    screen_dates = list(pd.date_range(start=start, end=end, freq="QS"))

if BENCH not in PX:
    print(f"WARNING: benchmark '{BENCH}' not found in bronze_yahoo_prices; Nifty columns will be empty. "
          f"Set benchmark_symbol in the config to the exact symbol stored there.")

print(f"Screen dates ({len(screen_dates)}): {[d.date().isoformat() for d in screen_dates]}")

# ------------------------------------------------------------
# 8. Run
# ------------------------------------------------------------
trade_rows, screen_rows = [], []

for D in screen_dates:
    df = build_features(D)
    if len(df) < MIN_UNIVERSE:
        print(f"{D.date()}: universe {len(df)} < {MIN_UNIVERSE}, skipped")
        continue

    n_q = int(df["qtr_eps_yoy"].notna().sum())
    print(f"{D.date()}: universe {len(df)}, with quarterly EPS YoY: {n_q}")

    for v in VARIANTS:
        if v["yearly_dates_only"] and D.month not in ANNUAL_MONTHS:
            continue
        exit_target = D + pd.Timedelta(days=v["hold"])
        b = bench_return(D, v["hold"])

        for style, picker in STYLES:
            picks = picker(df, v["use_qtr"])
            n_exit = 0
            for rank, (_, r) in enumerate(picks.iterrows(), start=1):
                sim = simulate(r["symbol"], r["entry_date"], r["entry_price"], exit_target)
                if sim is None:
                    continue
                exit_date, exit_px, reason, hold_ret = sim
                gross = exit_px / r["entry_price"] - 1
                net = gross - TXN_COST
                n_exit += 1
                trade_rows.append({
                    "variant": v["name"], "style": style,
                    "screen_date": D.date(), "entry_date": r["entry_date"].date(), "exit_date": exit_date.date(),
                    "symbol": r["symbol"], "company_name": r["company_name"], "industry": r["industry"],
                    "pick_rank": rank, "hold_days": v["hold"],
                    "entry_price": round(r["entry_price"], 2), "exit_price": round(exit_px, 2),
                    "exit_reason": reason,
                    "gross_return": gross, "net_return": net,
                    "hold_net_return": hold_ret - TXN_COST,       # same trade without stop/take-profit
                    "nifty_return": b,
                    "excess_return": (net - b) if b is not None else None,
                    "entry_cagr_3y": r["cagr_3y"], "entry_qtr_eps_yoy": r["qtr_eps_yoy"],
                    "entry_qtr_sales_yoy": r["qtr_sales_yoy"], "entry_pe": r["pe"],
                    "entry_ttm_yield": r["ttm_yield"], "entry_peg": r["peg"],
                    "run_id": RUN_ID,
                })

            screen_rows.append({
                "screen_date": D.date(), "variant": v["name"], "style": style,
                "universe_size": int(len(df)), "with_qtr_yoy": n_q,
                "eligible_picks": int(len(picks)), "picks_with_exit": int(n_exit),
                "run_id": RUN_ID,
            })

trades = pd.DataFrame(trade_rows)
screens = pd.DataFrame(screen_rows)
print(f"\nTotal trades: {len(trades)}")

# ------------------------------------------------------------
# 9. Summaries
# ------------------------------------------------------------
STR_COLS = {"variant", "style", "symbol", "company_name", "industry", "exit_reason", "run_id", "screened_at"}
DATE_COLS = {"screen_date", "entry_date", "exit_date"}

def to_spark(df, table):
    """Explicit schema so all-null columns never break createDataFrame."""
    fields = []
    for c in df.columns:
        if c in STR_COLS:
            t = StringType()
        elif c in DATE_COLS:
            t = DateType()
        elif pd.api.types.is_integer_dtype(df[c]):
            t = LongType()
        else:
            t = DoubleType()
        fields.append(StructField(c, t, True))

    def clean(v, t):
        if v is None:
            return None
        try:
            if pd.isna(v):
                return None
        except (TypeError, ValueError):
            pass
        if isinstance(t, StringType):
            return str(v)
        if isinstance(t, DateType):
            return pd.Timestamp(v).date()
        if isinstance(t, LongType):
            return int(v)
        f = float(v)
        return f if np.isfinite(f) else None

    recs = [tuple(clean(r[f.name], f.dataType) for f in fields) for r in df.to_dict("records")]
    sdf = spark.createDataFrame(recs, schema=StructType(fields))
    (sdf.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable(table))
    print(f"Saved {table}: {len(recs)} rows")

if trades.empty:
    print("No trades produced. Check backtest_v2_screens and the coverage prints above.")
    if not screens.empty:
        to_spark(screens, "backtest_v2_screens")
else:
    per_screen = (
        trades.groupby(["variant", "style", "screen_date"])
        .agg(screen_net=("net_return", "mean"), screen_nifty=("nifty_return", "first"), n=("symbol", "count"))
        .reset_index()
    )
    per_screen["screen_excess"] = per_screen["screen_net"] - per_screen["screen_nifty"]

    by_trade = (
        trades.groupby(["variant", "style"])
        .agg(
            n_picks=("symbol", "count"),
            avg_net_return=("net_return", "mean"),
            median_net_return=("net_return", "median"),
            win_rate=("net_return", lambda x: float((x > 0).mean())),
            stop_rate=("exit_reason", lambda x: float((x == "STOP").mean())),
            avg_hold_net_return=("hold_net_return", "mean"),
        )
        .reset_index()
    )
    by_screen = (
        per_screen.groupby(["variant", "style"])
        .agg(
            n_screens=("screen_date", "nunique"),
            avg_screen_net=("screen_net", "mean"),
            avg_screen_nifty=("screen_nifty", "mean"),
            avg_screen_excess=("screen_excess", "mean"),
            screens_beating_nifty=("screen_excess", lambda x: int((x > 0).sum())),
        )
        .reset_index()
    )
    summary = by_trade.merge(by_screen, on=["variant", "style"], how="left")

    a = summary[summary["variant"] == "ANNUAL_ONLY_90D"].set_index("style")
    q = summary[summary["variant"] == "QUARTERLY_GATED_90D"].set_index("style")
    keep = ["n_screens", "n_picks", "avg_net_return", "win_rate", "avg_screen_excess"]
    compare = a[keep].join(q[keep], lsuffix="_annual", rsuffix="_qtr", how="outer").reset_index()
    for c in ["n_picks", "avg_net_return", "win_rate"]:
        if f"{c}_annual" in compare and f"{c}_qtr" in compare:
            compare[f"delta_{c}"] = compare[f"{c}_qtr"] - compare[f"{c}_annual"]

    for df_ in (summary, compare, screens):
        df_["run_id"] = RUN_ID
        df_["screened_at"] = RUN_TS

    to_spark(trades, "backtest_v2_trades")
    to_spark(screens, "backtest_v2_screens")
    to_spark(summary, "backtest_v2_summary")
    to_spark(compare, "backtest_v2_compare")

    print("\n=== SUMMARY (variant x style) ===")
    display(spark.table("backtest_v2_summary"))
    print("\n=== QUARTERLY GATE vs ANNUAL ONLY (same dates, 90-day hold) ===")
    display(spark.table("backtest_v2_compare"))
    print("\n=== COVERAGE: how many screens could actually use the quarterly gate ===")
    display(spark.table("backtest_v2_screens").orderBy("screen_date", "variant", "style"))

print("\nExploratory and directional only. Small samples, survivorship bias, not investment advice.")

StatementMeta(, 109618b5-2497-4c09-a97f-97cfd858bd42, 3, Finished, Available, Finished, False)

/tmp/ipykernel_6230/3097749744.py:35: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_TS = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")


Universe: 343 | fundamentals: 27,755 | prices: 1,348,461 | last price: 2026-10-15
Screen dates (5): ['2025-07-01', '2025-10-01', '2026-01-01', '2026-04-01', '2026-07-01']
2025-07-01: universe 328, with quarterly EPS YoY: 0
2025-10-01: universe 331, with quarterly EPS YoY: 0
2026-01-01: universe 336, with quarterly EPS YoY: 0
2026-04-01: universe 340, with quarterly EPS YoY: 0
2026-07-01: universe 342, with quarterly EPS YoY: 206

Total trades: 22
Saved backtest_v2_trades: 22 rows
Saved backtest_v2_screens: 24 rows
Saved backtest_v2_summary: 3 rows
Saved backtest_v2_compare: 2 rows

=== SUMMARY (variant x style) ===


SynapseWidget(Synapse.DataFrame, 2c4df88d-7f23-430c-b126-bf045bd82fb0)


=== QUARTERLY GATE vs ANNUAL ONLY (same dates, 90-day hold) ===


SynapseWidget(Synapse.DataFrame, 2207ca9a-b83e-4a44-a44b-b025018171b0)


=== COVERAGE: how many screens could actually use the quarterly gate ===


SynapseWidget(Synapse.DataFrame, a752446a-5360-4572-aea2-38e753493d74)


Exploratory and directional only. Small samples, survivorship bias, not investment advice.
